# 💬 Exercícios — Processamento de Linguagem Natural (PLN)

**Disciplina:** Inteligência Artificial | **Nível:** Intermediário

> Pratique tokenização, análise de frequência, similaridade de textos e um chatbot simples baseado em regras.


## 1. Pré-processamento de Texto

In [ ]:
import re
from collections import Counter

texto = """
A Inteligência Artificial é uma área fascinante da Ciência da Computação.
A IA inclui Machine Learning, Redes Neurais e Processamento de Linguagem Natural.
O Machine Learning permite que computadores aprendam com dados.
As Redes Neurais são inspiradas no cérebro humano e são muito poderosas.
O PLN permite que computadores entendam e gerem linguagem humana.
"""

def preprocessar(texto):
    # 1. Minúsculas
    texto = texto.lower()
    # 2. Remove pontuação
    texto = re.sub(r'[^\w\s]', '', texto)
    # 3. Tokeniza
    tokens = texto.split()
    # 4. Remove stopwords (lista simplificada em português)
    stopwords = {'a','o','e','de','da','do','que','com','em','é','são','uma',
                 'um','no','na','os','as','as','se','por','para','não'}
    tokens = [t for t in tokens if t not in stopwords]
    return tokens

tokens = preprocessar(texto)
frequencias = Counter(tokens)

print(f"Total de tokens (após pré-processamento): {len(tokens)}")
print(f"Vocabulário único: {len(set(tokens))}")
print("\n20 palavras mais frequentes:")
for palavra, freq in frequencias.most_common(20):
    barra = '█' * freq
    print(f"  {palavra:<25} {freq:>3}  {barra}")


### 📝 Exercício 1

Modifique o texto com parágrafos sobre **outro tema** (ex: futebol, música, astronomia) e reanalise as frequências. O vocabulário muda significativamente? Qual palavra domina?

In [ ]:
meu_texto = """
# ✏️ Substitua por seu texto aqui
A astronomia é o estudo do universo e de seus corpos celestes.
Os planetas, estrelas e galáxias são objetos de estudo da astronomia.
O telescópio de Galileu revolucionou a observação do céu noturno.
"""
tokens_meu = preprocessar(meu_texto)
print("Frequências no meu texto:", Counter(tokens_meu).most_common(15))


## 2. TF-IDF — Importância das Palavras nos Documentos

In [ ]:
import math

documentos = [
    "aprendizado de máquina usa algoritmos para aprender com dados",
    "redes neurais são modelos inspirados no cérebro humano",
    "processamento de linguagem natural analisa textos humanos",
    "algoritmos genéticos imitam a evolução para otimização",
    "aprendizado profundo usa redes neurais com muitas camadas",
]

def calcular_tf(documento):
    tokens = documento.lower().split()
    total = len(tokens)
    return {t: tokens.count(t)/total for t in set(tokens)}

def calcular_idf(documentos):
    N = len(documentos)
    todas_palavras = set(w for doc in documentos for w in doc.lower().split())
    idf = {}
    for palavra in todas_palavras:
        docs_com_palavra = sum(1 for doc in documentos if palavra in doc.lower().split())
        idf[palavra] = math.log(N / (1 + docs_com_palavra))
    return idf

def tfidf(documento, documentos):
    tf = calcular_tf(documento)
    idf = calcular_idf(documentos)
    return {t: tf[t]*idf[t] for t in tf}

# Mostrar palavras mais importantes em cada documento
for i, doc in enumerate(documentos):
    scores = tfidf(doc, documentos)
    top = sorted(scores.items(), key=lambda x: -x[1])[:3]
    print(f"Doc {i+1}: {doc[:50]}...")
    print(f"  Palavras-chave: {', '.join(f'{p}({s:.3f})' for p,s in top)}\n")


### 📝 Exercício 2

Adicione um novo documento sobre **redes neurais convolucionais**. Quais palavras têm TF-IDF alto nesse documento? São diferentes das do documento 2 (sobre redes neurais)?

In [ ]:
docs_expandidos = documentos + [
    "redes neurais convolucionais processam imagens com filtros e pooling"
]
novo_doc = docs_expandidos[-1]
scores_novo = tfidf(novo_doc, docs_expandidos)
top_novo = sorted(scores_novo.items(), key=lambda x: -x[1])[:5]
print("Palavras-chave do novo documento:")
for p, s in top_novo:
    print(f"  {p:<25} TF-IDF={s:.4f}")


## 3. Similaridade de Cosseno entre Textos

In [ ]:
import numpy as np

def vetorizar(documentos):
    """Cria representação bag-of-words."""
    vocab = sorted(set(w for doc in documentos for w in doc.lower().split()))
    vocs = {w:i for i,w in enumerate(vocab)}
    matriz = np.zeros((len(documentos), len(vocab)))
    for i, doc in enumerate(documentos):
        for w in doc.lower().split():
            matriz[i][vocs[w]] += 1
    return matriz, vocab

def similaridade_cosseno(v1, v2):
    num = np.dot(v1, v2)
    den = np.linalg.norm(v1) * np.linalg.norm(v2)
    return num/den if den>0 else 0

matriz, vocab = vetorizar(documentos)

# Matriz de similaridade
n = len(documentos)
sim_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        sim_matrix[i][j] = similaridade_cosseno(matriz[i], matriz[j])

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7,6))
im = ax.imshow(sim_matrix, cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im)
ax.set_xticks(range(n)); ax.set_yticks(range(n))
ax.set_xticklabels([f'D{i+1}' for i in range(n)])
ax.set_yticklabels([f'D{i+1}' for i in range(n)])
for i in range(n):
    for j in range(n):
        ax.text(j,i,f'{sim_matrix[i,j]:.2f}',ha='center',va='center',fontsize=8)
ax.set_title('Similaridade de Cosseno entre Documentos')
plt.tight_layout(); plt.show()

# Documento mais similar ao D1
d0 = sim_matrix[0].copy(); d0[0] = 0
mais_similar = np.argmax(d0)
print(f"Documento mais similar ao D1: D{mais_similar+1}")
print(f"  D1: {documentos[0]}")
print(f"  D{mais_similar+1}: {documentos[mais_similar]}")


## 4. Chatbot Simples Baseado em Padrões

In [ ]:
import re

# Padrões e respostas para o chatbot
padroes_respostas = [
    (r"olá|oi|bom dia|boa tarde|boa noite",
     "Olá! Como posso ajudar você hoje?"),
    (r"o que é (inteligência artificial|ia)",
     "IA é a área que busca criar sistemas capazes de realizar tarefas que normalmente requerem inteligência humana!"),
    (r"o que é (machine learning|aprendizado de máquina)",
     "Machine Learning é um subcampo da IA onde os sistemas aprendem automaticamente a partir de dados."),
    (r"o que é (rede neural|redes neurais)",
     "Redes neurais são modelos computacionais inspirados no funcionamento do cérebro humano."),
    (r"tchau|até logo|até mais",
     "Até logo! Foi um prazer conversar com você. 😊"),
    (r"(.*)(obrigado|obrigada)(.*)",
     "De nada! Estou aqui para ajudar."),
    (r"quem é você|qual seu nome",
     "Sou um chatbot de IA criado para ajudar com dúvidas sobre Inteligência Artificial!"),
]

def chatbot(mensagem):
    mensagem = mensagem.lower().strip()
    for padrao, resposta in padroes_respostas:
        if re.search(padrao, mensagem):
            return resposta
    return "Interessante pergunta! Ainda estou aprendendo. Pode reformular?"

# Simulação de conversa
conversa = [
    "Olá!",
    "O que é inteligência artificial?",
    "O que é machine learning?",
    "Quem é você?",
    "Obrigado pela ajuda",
    "Tchau!",
    "Qual é o melhor algoritmo?",
]

print("=== Simulação de Chatbot ===")
for msg in conversa:
    print(f"Usuário: {msg}")
    print(f"Bot:     {chatbot(msg)}")
    print()


### �� Exercício Final

Adicione **5 novos padrões** ao chatbot para responder sobre:
1. Redes Neurais Convolucionais
2. PLN
3. Algoritmos Genéticos
4. Data Science
5. Uma pergunta pessoal (ex: "você tem sentimentos?")

Depois simule uma conversa usando todos os novos padrões.

In [ ]:
# ✏️ Estenda o chatbot:
meus_padroes = list(padroes_respostas) + [
    # TODO: adicione 5 novos padrões
    (r"o que é (pln|processamento de linguagem natural)", "TODO: sua resposta"),
    # ...
]

def meu_chatbot(mensagem):
    mensagem = mensagem.lower().strip()
    for padrao, resposta in meus_padroes:
        if re.search(padrao, mensagem):
            return resposta
    return "Não entendi. Pode reformular?"

# Teste seus novos padrões:
for msg in ["o que é pln", "você tem sentimentos?"]:
    print(f"Usuário: {msg}")
    print(f"Bot:     {meu_chatbot(msg)}\n")
